In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/twincheck'
F5C = f'{DRIVE}/F5_correctfiles'
CKPT = f'{F5C}/models'
F6 = f'{DRIVE}/F6'
ROOT = '/content/genimage_root'
REPO = '/content/twincheck'
!rm -rf {REPO}
!git clone -q https://github.com/himaallu/Group-Project.git {REPO}
%cd {REPO}
!cp -r {F5C}/src/vision/. src/vision/
!mkdir -p results/F5
!cp -r {F5C}/results/F5/. results/F5/
!nvidia-smi --query-gpu=name,memory.total --format=csv
os.makedirs(f'{F6}/cases', exist_ok=True)

In [ ]:
import pandas as pd
!mkdir -p {ROOT}/data/processed/vision
!ln -sfn {DRIVE}/data/processed/vision/genimage {ROOT}/data/processed/vision/genimage
for split in ('validation', 'test'):
    rows = pd.read_csv(f'data/processed/vision/genimage_{split}_manifest.csv')
    missing = [p for p in rows['exported_path'] if not os.path.exists(f'{ROOT}/{p}')]
    print(split, len(rows), rows.groupby('generator').size().to_dict(), 'missing', len(missing))
    assert not missing
!ls -la {CKPT}
assert all(os.path.exists(f'{CKPT}/{a}_s42.pt') for a in ('resnet50', 'vit_b16'))

In [ ]:
%cd /content/twincheck


In [ ]:
import shutil
import numpy as np
import torch
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from src.vision.detectors import build_model, eval_transform, manifest_rows
from src.vision.train_detector import metrics

LOCAL = '/content/local_root'
device = 'cuda'
ARCHS = ('resnet50', 'vit_b16')
splits = {s: manifest_rows(s) for s in ('validation', 'test')}
paths = sorted({p for r in splits.values() for p in r['exported_path']})


def fetch(p):
    dst = Path(LOCAL) / p
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(f'{ROOT}/{p}', dst)


with ThreadPoolExecutor(32) as pool:
    list(tqdm(pool.map(fetch, paths), total=len(paths), desc='copy to local disk'))


class Imgs(Dataset):
    def __init__(self, rows):
        self.paths = list(rows['exported_path'])
        self.tf = eval_transform()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(f'{LOCAL}/{self.paths[i]}').convert('RGB')
        return self.tf(img), img.size[0], img.size[1]


nets = {}
for arch in ARCHS:
    net = build_model(arch, pretrained=False).to(device)
    net.load_state_dict(torch.load(f'{CKPT}/{arch}_s42.pt', map_location=device)['model'])
    nets[arch] = net.eval()

parts, check = [], []
for split, rows in splits.items():
    loader = DataLoader(Imgs(rows), batch_size=128, num_workers=4)
    probs = {a: [] for a in ARCHS}
    widths, heights = [], []
    with torch.no_grad():
        for x, w, h in tqdm(loader, desc=split):
            x = x.to(device)
            for arch, net in nets.items():
                with torch.autocast(device_type='cuda'):
                    logits = net(x).squeeze(1)
                probs[arch].append(torch.sigmoid(logits.float()).cpu().numpy())
            widths.append(w.numpy())
            heights.append(h.numpy())
    table = rows[['md5', 'split', 'generator', 'class_name', 'exported_path']].copy()
    table['width'] = np.concatenate(widths)
    table['height'] = np.concatenate(heights)
    table['format'] = [Path(p).suffix.lower().lstrip('.') for p in rows['exported_path']]
    y = (rows['class_name'] == 'fake').astype(int).to_numpy()
    for arch in ARCHS:
        p_ai = np.concatenate(probs[arch])
        table[f'p_ai_{arch}'] = p_ai
        check.append({'model': arch, 'split': split, **metrics(p_ai, y, rows)})
    parts.append(table)
pred = pd.concat(parts, ignore_index=True)
pred.to_csv(f'{F6}/predictions.csv', index=False)

cols = ['accuracy', 'precision', 'recall', 'f1', 'auroc']
ref = pd.read_csv(f'{F5C}/results/F5/cnn_vit.csv').query("corruption == 'clean'")[['model', 'split'] + cols]
cmp = pd.DataFrame(check)[['model', 'split'] + cols].merge(ref, on=['model', 'split'], suffixes=('', '_f5'))
cmp['max_abs_diff'] = np.abs(cmp[cols].to_numpy() - cmp[[c + '_f5' for c in cols]].to_numpy()).max(axis=1)
cmp.to_csv(f'{F6}/check_against_f5.csv', index=False)
print(cmp.round(4).to_string())


In [ ]:
test = pred[pred['split'] == 'test'].copy()
P = ['p_ai_resnet50', 'p_ai_vit_b16']
sd = test[test['class_name'] == 'fake']
real = test[test['class_name'] == 'real']
both_miss = sd[(sd[P] < 0.5).all(axis=1)]
vit_only = sd[(sd['p_ai_resnet50'] < 0.5) & (sd['p_ai_vit_b16'] >= 0.5)]
resnet_only = sd[(sd['p_ai_resnet50'] >= 0.5) & (sd['p_ai_vit_b16'] < 0.5)]
print('SD missed by both', len(both_miss), '| caught by ViT only', len(vit_only), '| caught by ResNet only', len(resnet_only))
print('real false positives: resnet50', int((real['p_ai_resnet50'] >= 0.5).sum()), '| vit_b16', int((real['p_ai_vit_b16'] >= 0.5).sum()))
assert len(both_miss) >= 4 and len(vit_only) >= 2
picks = [
    ('confident miss, both models', both_miss.loc[both_miss[P].mean(axis=1).nsmallest(2).index]),
    ('borderline miss, both models', both_miss.loc[both_miss[P].max(axis=1).nlargest(2).index]),
    ('caught by ViT-B/16 only', vit_only.nlargest(2, 'p_ai_vit_b16')),
    ('false positive on real', real.loc[real[P].max(axis=1).nlargest(2).index]),
]
cases = pd.concat([part.assign(case=name) for name, part in picks], ignore_index=True)
cases.insert(0, 'case_id', range(1, len(cases) + 1))
for arch in ARCHS:
    cases[f'correct_{arch}'] = (cases[f'p_ai_{arch}'] >= 0.5) == (cases['class_name'] == 'fake')
print(cases[['case_id', 'case', 'generator', 'width', 'height', 'format'] + P + ['correct_resnet50', 'correct_vit_b16']].round(4).to_string())

In [ ]:
import shutil
import matplotlib.pyplot as plt
from IPython.display import Image as show_image, display
from src.vision.detectors import GenImageDataset
from src.vision.gradcam import grad_cam, unnormalise

data = GenImageDataset(cases.assign(image_path=cases['exported_path']), ROOT, eval_transform())
x = torch.stack([data[i][0] for i in range(len(data))]).to(device)
cams = {}
for arch, net in nets.items():
    cams[arch], cases[f'p_ai_gradcam_{arch}'] = grad_cam(net, arch, x)
cases.to_csv(f'{F6}/cases.csv', index=False)
for _, row in cases.iterrows():
    shutil.copy(f"{ROOT}/{row['exported_path']}", f"{F6}/cases/case{row['case_id']}_{row['md5']}{Path(row['exported_path']).suffix}")

names = {'resnet50': 'ResNet-50', 'vit_b16': 'ViT-B/16'}
fig, axes = plt.subplots(3, 8, figsize=(24, 10))
for i, row in cases.iterrows():
    axes[0, i].imshow(unnormalise(x[i]))
    axes[0, i].set_title(f"Case {row['case_id']}: {row['generator']}\n{row['case']}", fontsize=8)
    for r, arch in enumerate(ARCHS, start=1):
        axes[r, i].imshow(unnormalise(x[i]))
        axes[r, i].imshow(cams[arch][i], cmap='jet', alpha=0.4)
        axes[r, i].set_title(f"{names[arch]}  P(AI) = {row[f'p_ai_{arch}']:.3f}", fontsize=8)
for ax in axes.flat:
    ax.axis('off')
fig.suptitle('Grad-CAM on eight test cases (seed 42, threshold 0.5)')
fig.tight_layout()
fig.savefig(f'{F6}/fig_f6_error_cases.png', dpi=200)
plt.close(fig)
display(show_image(f'{F6}/fig_f6_error_cases.png', width=1600))
print(cases[['case_id', 'case', 'md5', 'p_ai_resnet50', 'p_ai_gradcam_resnet50', 'p_ai_vit_b16', 'p_ai_gradcam_vit_b16']].round(4).to_string())


In [ ]:
print(cases[['case_id', 'case', 'md5'] + P + ['p_ai_gradcam_resnet50', 'p_ai_gradcam_vit_b16']].round(4).to_string())
!ls -la {F6} {F6}/cases